# Chạy QG-RSIAT trên Kaggle

Chọn **GPU** trong Notebook options và thêm Kaggle Dataset chứa dữ liệu cùng checkpoint pretrained ViT vào **Notebook > Input**. Trong cell **Nối dataset và checkpoint**, chọn dataset rồi điền đường dẫn `DATASET_DIR` và `VIT_CHECKPOINT`.

CIFAR-100 cần thư mục `cifar-100-python` gồm `train`, `test`, `meta`; ImageNet-A/R cần `train/<class>` và `test/<class>`. Dùng checkpoint pretrained ViT-B/16 ImageNet-21k tương thích với `timm==0.6.12` (`ViT-B_16.npz`). Chạy các cell theo thứ tự từ trên xuống. Log được lưu trong `/kaggle/working/QG_RSIAT/logs/`.

In [ ]:
!git clone --depth 1 https://github.com/deety03503/QG_RSIAT.git /kaggle/working/QG_RSIAT

In [ ]:
!python -m pip install --quiet "timm==0.6.12" "easydict==1.13"
!nvidia-smi
!python -m pip show torch torchvision timm easydict

## Nối dataset và checkpoint

Chọn `DATASET`, rồi đặt `DATASET_DIR` thành đường dẫn chính xác tới thư mục dữ liệu trong `/kaggle/input`. Với CIFAR-100, dùng thư mục `cifar-100-python`; cell bên dưới sẽ sao chép dữ liệu vào vùng ghi được vì Kaggle Input chỉ đọc. Với ImageNet-A/R, dùng thư mục có `train/<class>` và `test/<class>`; cell sẽ nối trực tiếp để tránh sao chép bộ dữ liệu lớn. Đặt `VIT_CHECKPOINT` tới file pretrained ViT-B/16 ImageNet-21k `ViT-B_16.npz`.

In [ ]:
# Chọn dataset và thay đường dẫn mẫu bằng đường dẫn trong /kaggle/input.
DATASET = "cifar224"  # cifar224, imageneta hoặc imagenetr
DATASET_DIR = "/kaggle/input/your-dataset/cifar-100-python"
VIT_CHECKPOINT = "/kaggle/input/your-checkpoint/ViT-B_16.npz"

DATASET_NAME = {"cifar224": "cifar-100-python", "imageneta": "imagenet-a", "imagenetr": "imagenet-r"}[DATASET]
CONFIG_NAME = f"adapter_{DATASET}"

from pathlib import Path
import shutil

dataset_target = Path(f"/kaggle/working/QG_RSIAT/data/datasets/{DATASET_NAME}")
dataset_target.parent.mkdir(parents=True, exist_ok=True)
if dataset_target.is_symlink():
    dataset_target.unlink()
if DATASET == "cifar224":
    shutil.copytree(DATASET_DIR, dataset_target, dirs_exist_ok=True)
else:
    dataset_target.symlink_to(DATASET_DIR, target_is_directory=True)

!mkdir -p /root/.cache/torch/hub/checkpoints
!ln -sfn "{VIT_CHECKPOINT}" /root/.cache/torch/hub/checkpoints/ViT-B_16.npz
print(f"Connected {DATASET_DIR} to writable path {dataset_target}")

## CONFIG OVERRIDE

Cell tiếp theo sao chép `adapter_{DATASET}.json` thành config Kaggle riêng và áp dụng thiết lập GPU, số worker, seed. Config gốc không bị thay đổi.

In [ ]:
import json
from pathlib import Path

base_config = Path(f"/kaggle/working/QG_RSIAT/exps/{CONFIG_NAME}.json")
config = json.loads(base_config.read_text())
config.update({"device": ["0"], "num_worker": 2, "seed": [1993]})
output_config = base_config.with_name(f"{CONFIG_NAME}.kaggle.json")
output_config.write_text(json.dumps(config, indent=2) + "\n")
print(f"Config ready: {output_config}")

In [ ]:
%cd /kaggle/working/QG_RSIAT
!python main.py --config "exps/{CONFIG_NAME}.kaggle.json"